# Práctica: BigQuery sin BigQuery

**Sesión 25 · Pedidos Express**

En el lab de la sesión anterior usaste BigQuery para hacer preguntas a una tabla de facturación. Aquí harás **las mismas consultas SQL**, pero en tu computadora (o en Google Colab), con Python y **DuckDB**: un motor SQL que corre dentro de Python y, como BigQuery, lee los datos **por columnas**.

Los datos: **1.2 millones de pedidos** de Pedidos Express entre 2023 y 2025. Son más filas de las que caben en una hoja de Excel (1 048 576).

| En el lab (BigQuery) | En esta práctica (Python + DuckDB) |
| --- | --- |
| Archivo en Cloud Storage | `ventas.csv` en tu carpeta |
| Conjunto de datos `billing_dataset`, tabla `sampleinfotable` | Tabla `ventas` |
| Esquema, Detalles y Vista previa | `DESCRIBE`, `COUNT(*)` y `LIMIT 5` |
| `WHERE cost > 0` | `WHERE monto > 0` |
| `COUNT(*)` y `SUM(cost)` por servicio | `COUNT(*)` y `SUM(monto)` por categoría |

Ejecuta las celdas **en orden**, de arriba hacia abajo (Shift + Enter).

## Paso 0. Instalar DuckDB

In [ ]:
%pip install -q duckdb

## Paso 1. Generar los datos

La primera celda guarda el programa `generar_ventas.py`; la segunda lo ejecuta y crea `ventas.csv` (unos 105 MB; tarda menos de un minuto).

Los datos son inventados, pero salen **iguales para todos** porque el programa usa una semilla fija.

In [ ]:
%%writefile generar_ventas.py
"""Genera ventas.csv: 1.2 millones de pedidos de Pedidos Express (2023-2025).

Los datos son inventados, pero siempre salen iguales (semilla fija),
asi todos en clase obtienen los mismos resultados.

Uso:
    python generar_ventas.py
"""

import csv
import random
from datetime import date, timedelta

FILAS = 1_200_000
ARCHIVO = "ventas.csv"

PRODUCTOS = {
    "Electronica": {"Audifonos": 89.0, "Cargador": 45.0, "Mouse": 59.0, "Teclado": 129.0, "Parlante": 199.0},
    "Hogar": {"Lampara": 79.0, "Sarten": 99.0, "Juego de sabanas": 149.0},
    "Supermercado": {"Arroz 5 kg": 24.0, "Aceite": 12.0, "Cafe": 32.0, "Leche x6": 28.0},
    "Moda": {"Polo": 49.0, "Zapatillas": 249.0, "Casaca": 189.0},
    "Farmacia": {"Paracetamol": 8.0, "Protector solar": 59.0, "Vitaminas": 69.0},
}
PESO_CATEGORIA = {"Electronica": 18, "Hogar": 14, "Supermercado": 38, "Moda": 16, "Farmacia": 14}

DISTRITOS = ["Miraflores", "San Isidro", "Surco", "La Molina", "San Borja",
             "Barranco", "Jesus Maria", "Lince", "Magdalena", "Pueblo Libre"]
PESO_DISTRITO = [14, 10, 18, 9, 9, 6, 9, 8, 8, 9]

PAGOS = ["Tarjeta", "Yape", "Efectivo", "Plin"]
PESO_PAGO = [45, 30, 15, 10]

# Diciembre y julio venden mas (fiestas y gratificacion).
PESO_MES = {1: 0.9, 2: 0.85, 3: 0.95, 4: 0.95, 5: 1.0, 6: 1.0,
            7: 1.25, 8: 1.0, 9: 0.95, 10: 1.0, 11: 1.1, 12: 1.45}


def main():
    random.seed(24)

    # Cada dia de 2023 a 2025 tiene un peso: la empresa crece con el tiempo.
    dias = []
    pesos_dia = []
    inicio, fin = date(2023, 1, 1), date(2025, 12, 31)
    total_dias = (fin - inicio).days + 1
    for i in range(total_dias):
        dia = inicio + timedelta(days=i)
        dias.append(dia.isoformat())
        pesos_dia.append((1 + i / total_dias) * PESO_MES[dia.month])

    fechas = sorted(random.choices(dias, weights=pesos_dia, k=FILAS))
    categorias = random.choices(list(PESO_CATEGORIA), weights=list(PESO_CATEGORIA.values()), k=FILAS)
    distritos = random.choices(DISTRITOS, weights=PESO_DISTRITO, k=FILAS)
    pagos = random.choices(PAGOS, weights=PESO_PAGO, k=FILAS)
    cantidades = random.choices([1, 2, 3, 4, 5], weights=[55, 25, 11, 6, 3], k=FILAS)
    cancelados = random.choices([False, True], weights=[92, 8], k=FILAS)

    with open(ARCHIVO, "w", newline="", encoding="utf-8") as f:
        escritor = csv.writer(f)
        escritor.writerow(["pedido_id", "fecha", "cliente_id", "distrito", "categoria", "producto",
                           "cantidad", "precio_unitario", "monto", "metodo_pago", "estado"])
        for i in range(FILAS):
            categoria = categorias[i]
            producto = random.choice(list(PRODUCTOS[categoria]))
            precio = PRODUCTOS[categoria][producto]
            cancelado = cancelados[i]
            # Un pedido cancelado no se cobra: su monto es 0.
            monto = 0.0 if cancelado else round(precio * cantidades[i], 2)
            escritor.writerow([
                f"P-{i + 1:07d}", fechas[i], f"C-{random.randint(1, 50_000):05d}",
                distritos[i], categoria, producto, cantidades[i], precio, monto,
                pagos[i], "cancelado" if cancelado else "entregado",
            ])

    print(f"Listo: {ARCHIVO} con {FILAS:,} filas.")


if __name__ == "__main__":
    main()

In [ ]:
%run generar_ventas.py

## Paso 2. Crear la tabla y explorarla

Como en el lab: primero se crea la tabla a partir del archivo; luego se mira su **esquema** (columnas y tipos), sus **detalles** (cuántas filas) y una **vista previa**.

In [ ]:
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE ventas AS SELECT * FROM 'ventas.csv'")
print("Tabla ventas creada")

**Esquema:** qué columnas tiene la tabla y de qué tipo es cada una.

In [ ]:
duckdb.sql("DESCRIBE ventas")

**Detalles:** cuántas filas tiene.

In [ ]:
duckdb.sql("SELECT COUNT(*) AS filas FROM ventas")

**Vista previa:** las primeras 5 filas.

In [ ]:
duckdb.sql("SELECT * FROM ventas LIMIT 5")

## Paso 3. Una consulta simple

En el lab filtraste las filas con `cost > 0`. Aquí un pedido **cancelado** no se cobra y tiene `monto = 0`. Esta consulta muestra solo pedidos cobrados.

In [ ]:
duckdb.sql("""
    SELECT pedido_id, fecha, categoria, producto, monto
    FROM ventas
    WHERE monto > 0
    LIMIT 10
""")

## Paso 4. Preguntas de negocio

Cada celda responde una pregunta que haría el gerente. Lee la consulta antes de ejecutarla y trata de adivinar qué saldrá.

**4.1 ¿Cuáles fueron los últimos 10 pedidos cobrados?** `ORDER BY ... DESC` ordena de mayor a menor; `LIMIT` se queda con los primeros.

In [ ]:
duckdb.sql("""
    SELECT pedido_id, fecha, distrito, producto, monto
    FROM ventas
    WHERE monto > 0
    ORDER BY fecha DESC, pedido_id DESC
    LIMIT 10
""")

**4.2 ¿Cuántos pedidos pasaron de 500 soles?**

In [ ]:
duckdb.sql("""
    SELECT COUNT(*) AS pedidos_grandes
    FROM ventas
    WHERE monto > 500
""")

**4.3 ¿Qué categoría tiene más pedidos?** `GROUP BY` junta las filas que tienen la misma categoría y `COUNT(*)` cuenta cuántas hay en cada grupo.

In [ ]:
duckdb.sql("""
    SELECT categoria, COUNT(*) AS pedidos
    FROM ventas
    GROUP BY categoria
    ORDER BY pedidos DESC
""")

**4.4 ¿Qué método de pago se usa más en los pedidos cobrados?**

In [ ]:
duckdb.sql("""
    SELECT metodo_pago, COUNT(*) AS pedidos
    FROM ventas
    WHERE monto > 0
    GROUP BY metodo_pago
    ORDER BY pedidos DESC
""")

**4.5 ¿Qué categoría vende más dinero?** Es la misma consulta que cierra el lab: `SUM` suma los montos de cada grupo y `ROUND(..., 2)` deja dos decimales.

In [ ]:
duckdb.sql("""
    SELECT categoria, ROUND(SUM(monto), 2) AS total_soles
    FROM ventas
    GROUP BY categoria
    ORDER BY total_soles DESC
""")

Compara con 4.3: ¿la categoría con **más pedidos** es la que **más dinero** vende?

## Paso 5. ¿Por qué leer por columnas es más rápido?

BigQuery guarda cada tabla **por columnas**: si la consulta solo usa `categoria` y `monto`, solo lee esas dos columnas.

Para verlo, guarda la misma tabla en dos formatos y ejecuta la misma consulta sobre cada uno:

- **CSV:** guarda fila por fila. Para leer `monto` hay que recorrer cada línea completa.
- **Parquet:** guarda columna por columna, como BigQuery. Puede leer solo `categoria` y `monto`.

In [ ]:
import os
import time

duckdb.sql("COPY ventas TO 'ventas.parquet' (FORMAT parquet)")

for archivo in ["ventas.csv", "ventas.parquet"]:
    inicio = time.time()
    duckdb.sql(f"SELECT categoria, SUM(monto) FROM '{archivo}' GROUP BY categoria").fetchall()
    segundos = time.time() - inicio
    megas = os.path.getsize(archivo) / 1_000_000
    print(f"{archivo:15} {megas:6.1f} MB   {segundos:.3f} s")

El archivo Parquet ocupa menos y la consulta termina antes, aunque las dos tienen exactamente los mismos datos.

## Retos

Escribe tú la consulta. Usa como modelo las del Paso 4.

Para sacar el año o el mes de una fecha se usa `EXTRACT(YEAR FROM fecha)` o `EXTRACT(MONTH FROM fecha)`; funciona igual en BigQuery. Para el promedio se usa `AVG(columna)`.

**Reto 1.** ¿Cuánto se vendió (en soles) cada año?

In [ ]:
duckdb.sql("""
    -- Escribe tu consulta aquí
    SELECT 1
""")

**Reto 2.** ¿Cuáles fueron los 3 distritos que más compraron (en soles) en 2025?

In [ ]:
duckdb.sql("""
    -- Escribe tu consulta aquí
    SELECT 1
""")

**Reto 3.** Sumando los tres años, ¿qué mes del año vende más?

In [ ]:
duckdb.sql("""
    -- Escribe tu consulta aquí
    SELECT 1
""")

**Reto 4.** En los pedidos cobrados, ¿cuál es el monto promedio de un pedido en cada categoría? ¿Explica esto el resultado de 4.3 y 4.5?

In [ ]:
duckdb.sql("""
    -- Escribe tu consulta aquí
    SELECT 1
""")

**Reto 5.** ¿Cuántos pedidos se cancelaron y qué porcentaje del total son?

In [ ]:
duckdb.sql("""
    -- Escribe tu consulta aquí
    SELECT 1
""")